# Load Packages

In [ ]:
%load_ext autoreload
%autoreload 2

# Important libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from os.path import join
import joblib
import sys
import torch
import pickle

sys.path.append("../../")

from src.configs.default_configs import fn_model, fn_pred, fn_pred_perf, device
from src.configs.crc_config import data_name
from src.file_manager.filepath import FilePath
from src.models.gpc.train import train_gpc
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score, precision_score, f1_score
from src.evaluation.perf_metrics import get_bce_from_pred_prob
from seed_file import seed
# seed = 2024

batch_size = 32
eval_batch_size = 128

tuning_seed = 2024

fp = FilePath(data_name=data_name, seed=seed)
fp_scaler_file = join(fp.get_preprocessed_folder(), f"minmax_scaler.pickle")
fp_encoder_file = join(fp.get_preprocessed_folder(), f"encoder.pickle")
fp_split_dict_file = join(fp.get_preprocessed_folder(), f"split_dict.joblib")
fp_split_dict_oversampled_file = join(fp.get_preprocessed_folder(), f"split_dict_oversampled.joblib")

# Load Data

In [ ]:
split_dict_scaled = joblib.load(fp_split_dict_file)
feat_cols_w_pc = ['Age_interview', 'PC1', 'PC2', 'PC3', 'BMI', 'telomere length', 'aHEI2010score', 'aMED', 'DASH', 'SBP', 'DBP', 'Leisure screen time', 'z_pgs000055', 'z_pgs000734', 'Sex (0=Male, 1=Female)', 'alcohol_DailyandWeekly(1)vsMonthlyandNonDrinkers(0)', 'smoke_ex(1)', 'smoke_current(2)', 'Prevalent_diabetes']
target_col = "colorectal cancer"

# Model Training (GPC)

In [ ]:
fp_model = join(fp.get_parent_folder(fn_model), "gpc.pkl")
test_pred_df_gpc = train_gpc(
    split_dict=split_dict_scaled, feat_cols=feat_cols_w_pc, target_col=target_col, 
    fp_model=fp_model, seed=seed)
display(test_pred_df_gpc)
fp_gpc_predictions_file = join(fp.get_parent_folder(fn_pred), "gpc.csv")
test_pred_df_gpc.to_csv(fp_gpc_predictions_file)

# Get Prediction Performance

In [ ]:
fp_gpc_predictions_file = join(fp.get_parent_folder(fn_pred), "gpc.csv")
test_pred_df_gpc = pd.read_csv(fp_gpc_predictions_file, index_col=0)
len_test = len(split_dict_scaled["test_df"])
model_perf_list= []
fp_gpc_predictions_file = join(fp.get_parent_folder(fn_pred), "gpc.csv")
pred_df_gpc = pd.read_csv(fp_gpc_predictions_file, index_col=0).iloc[-len_test:]
fp_perf_evaluation = join(fp.get_parent_folder(fn_pred_perf), "gpc.csv")
y_true = pred_df_gpc[target_col]
y_score = pred_df_gpc[target_col+"_pred_prob_gpc"]
y_pred = pred_df_gpc[target_col+"_pred_label_gpc"]
model_perf_list.append({
    "AUC": roc_auc_score(y_true=y_true, y_score=y_score),
    "Accuracy": accuracy_score(y_true=y_true, y_pred=y_pred), 
    "Crossentropy Loss": np.mean(get_bce_from_pred_prob(y_true, y_score)),
    "Recall": recall_score(y_true=y_true, y_pred=y_pred), 
    "Precision": precision_score(y_true=y_true, y_pred=y_pred),
    "F1-Score": f1_score(y_true=y_true, y_pred=y_pred)
})
perf_df = pd.DataFrame(model_perf_list)
perf_df.index = ["Test"]
display(perf_df)
perf_df.to_csv(fp_perf_evaluation)